# Wayline GGUF export (minimal, reliable)

Produces a merged, `Q4_K_M`-quantized GGUF of the Wayline distractor model for the
live game demo. This is the streamlined path: it merges the released LoRA adapter
into the Qwen3-4B base, converts to GGUF, and quantizes — skipping the heavy
provenance/reviewed-cache audits that are only needed for the fully-packaged
production runtime.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
You will be asked for your Hugging Face token once.

In [ ]:
%%capture
# Pin mutually-compatible stable versions. Unbounded upgrades pulled Transformers 5.x,
# which writes tokenizer metadata that its own Qwen loader cannot read.
!pip install -q -U "transformers==4.53.3" "peft==0.16.0" accelerate "huggingface_hub<1.0" sentencepiece protobuf
# Colab ships an old torchao that peft rejects; it is not used for a bf16 merge.
!pip uninstall -y torchao

In [ ]:
from getpass import getpass
from huggingface_hub import login

# Full-precision base to merge onto (public). The adapter was QLoRA-trained on the
# bnb-4bit repack of this same model; LoRA merges onto the 16-bit base by module name.
BASE_ID = "Qwen/Qwen3-4B"
ADAPTER_ID = "j2ampn/qwen3-4b-distractor-lora-v7"
OUT_NAME = "wayline_qwen3_4b_q4_k_m.gguf"

login(getpass("Hugging Face token: ").strip())
print("base:", BASE_ID, "| adapter:", ADAPTER_ID)

In [ ]:
import os, shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("loading base (bf16)...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(BASE_ID, trust_remote_code=True)

print("applying + merging adapter...")
model = PeftModel.from_pretrained(base, ADAPTER_ID)
model = model.merge_and_unload()

MERGED = "/content/wayline-merged"
shutil.rmtree(MERGED, ignore_errors=True)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Free disk: {free_gb:.1f} GB")
assert free_gb >= 12, "Not enough free disk for merged model."
print("Saving ~8 GB in 1 GB shards. This may show no progress for 5–15 minutes. DO NOT INTERRUPT.")
model.save_pretrained(MERGED, safe_serialization=True, max_shard_size="1GB")
tokenizer.save_pretrained(MERGED)

# Transformers 5.x can serialize `extra_special_tokens` as a list even though its
# Qwen loader expects a mapping. Remove that optional generated field; the real
# token definitions remain in tokenizer.json / added_tokens_decoder.
import json
config_path = os.path.join(MERGED, "tokenizer_config.json")
with open(config_path, encoding="utf-8") as f:
    tokenizer_config = json.load(f)
tokenizer_config.pop("extra_special_tokens", None)
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(tokenizer_config, f, ensure_ascii=False, indent=2)
    f.write("\n")

# Fail here (cheaply) rather than after an 8 GB conversion if metadata is bad.
AutoTokenizer.from_pretrained(MERGED, trust_remote_code=True)
print("MERGED MODEL + TOKENIZER SAVED:", sorted(os.listdir(MERGED)))

In [ ]:
%%capture
# Clone llama.cpp at the exact pinned commit and build the quantizer (CPU is fine).
import os
PIN = "6b4dc2116a92c5c8f2782bfe51fabe5ee66fb5ef"
if not os.path.isdir("/content/llama.cpp"):
    !git clone https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!cd /content/llama.cpp && git fetch --depth 1 origin $PIN && git checkout $PIN
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /content/cmake_cfg.log 2>&1
!cd /content/llama.cpp && cmake --build build --target llama-quantize -j 4 > /content/cmake_build.log 2>&1

In [ ]:
# Verify tokenizer + quantizer, then convert (f16) and quantize (Q4_K_M).
# subprocess(check=True) stops immediately on the real error instead of cascading.
import os, subprocess
from transformers import AutoTokenizer

MERGED = "/content/wayline-merged"
F16 = "/content/model-f16.gguf"
FINAL = "/content/wayline_qwen3_4b_q4_k_m.gguf"
QUANT = "/content/llama.cpp/build/bin/llama-quantize"
assert os.path.exists(QUANT), "llama-quantize did not build; see /content/cmake_build.log"
AutoTokenizer.from_pretrained(MERGED, trust_remote_code=True)
for partial in (F16, FINAL):
    if os.path.exists(partial):
        os.remove(partial)

subprocess.run([
    "python", "/content/llama.cpp/convert_hf_to_gguf.py", MERGED,
    "--outfile", F16, "--outtype", "f16",
], check=True)
assert os.path.getsize(F16) > 1_000_000_000, "F16 GGUF is unexpectedly small"

subprocess.run([QUANT, F16, FINAL, "Q4_K_M"], check=True)
size_gb = os.path.getsize(FINAL) / 1e9
print(f"GGUF ready: {FINAL}  ({size_gb:.2f} GB)")

In [ ]:
import hashlib
h = hashlib.sha256()
with open("/content/wayline_qwen3_4b_q4_k_m.gguf", "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        h.update(chunk)
print("sha256:", h.hexdigest())

In [ ]:
# Save to Google Drive (recommended for a ~2.5 GB file), then share the path.
from google.colab import drive
drive.mount("/content/drive")
import shutil, os
dest_dir = "/content/drive/MyDrive/wayline"
os.makedirs(dest_dir, exist_ok=True)
shutil.copy2("/content/wayline_qwen3_4b_q4_k_m.gguf", f"{dest_dir}/wayline_qwen3_4b_q4_k_m.gguf")
print("copied to", f"{dest_dir}/wayline_qwen3_4b_q4_k_m.gguf")
print("Download it from Drive, then give the agent the local path to the .gguf.")

In [ ]:
# Optional alternative: direct browser download (may be slow/unreliable for GBs).
# from google.colab import files
# files.download("/content/wayline_qwen3_4b_q4_k_m.gguf")